# RevProof experiments

Choose a Colab GPU runtime. Upload `revproof.zip` in cell 3. The included toy and two-question Flask pilot are **wiring checks**, not training data. Read `COLAB_RUNBOOK.md` for annotation and reporting rules.


In [ ]:
!nvidia-smi


In [ ]:
from google.colab import files
import zipfile
uploaded = files.upload()  # choose revproof.zip
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as zf:
    zf.extractall('/content')
%cd /content/revproof
!pip -q install -r requirements-colab.txt


In [ ]:
import torch, transformers, peft, trl
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('torch/transformers/peft/trl:', torch.__version__, transformers.__version__, peft.__version__, trl.__version__)
!python -m unittest discover -p test_pipeline.py -v
!python evaluate_generator.py --data data/flask_pilot_audited.jsonl --split dev --template-only --output runs/flask_template.jsonl


## Optional GPU smoke run (fictional data)

Run this once to check model download, 4-bit training, adapter saving, and inference. Its scores must **never** appear as empirical evidence in the report.


In [ ]:
!python make_toy_data.py
!python train_generator.py --data data/toy.jsonl --output runs/toy_sft --epochs 1 --allow-synthetic-smoke
!python evaluate_generator.py --data data/toy.jsonl --split dev --adapter runs/toy_sft/adapter --output runs/toy_sft_dev.jsonl


## Gate 1 — upload audited real revisions

Prepare `real.jsonl` as specified in `README.md`. At least train and dev must contain different independent revision groups. Reserve test until model selection is complete.


In [ ]:
uploaded = files.upload()  # choose your audited real.jsonl
assert 'real.jsonl' in uploaded, 'Upload real.jsonl after manual audit'
from pathlib import Path
Path('data/real.jsonl').write_bytes(uploaded['real.jsonl'])
from collections import Counter
from revproof_data import read_records
rows = read_records('data/real.jsonl')
assert all(r['source_type'] == 'real' for r in rows)
print('questions by split:', Counter(r['split'] for r in rows))
print('independent groups:', {s: len({r['group'] for r in rows if r['split']==s}) for s in ('train','dev','test')})
print('edit types:', Counter(r['edit_type'] for r in rows))
assert {r['split'] for r in rows} >= {'train','dev'}


## Gate 2 — SFT and same-backbone baselines

The 1.5B model is the lower-memory pilot. If your GPU can fit it, change `MODEL` to `Qwen/Qwen3-4B-Instruct-2507` **before** running any model comparison, then keep it fixed.


In [ ]:
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
import subprocess
def run(*args):
    print('RUN:', ' '.join(map(str, args)))
    subprocess.run(['python', *map(str, args)], check=True)
run('train_generator.py', '--data','data/real.jsonl','--output','runs/sft','--model',MODEL)


In [ ]:
run('evaluate_generator.py','--data','data/real.jsonl','--split','dev','--template-only','--output','runs/template_dev.jsonl')
for name, adapter, n, filt in [
    ('prompt_dev',None,1,False),('sft_dev','runs/sft/adapter',1,False),
    ('prompt_filter_dev',None,4,True),('sft_filter_dev','runs/sft/adapter',4,True)]:
    args=['evaluate_generator.py','--data','data/real.jsonl','--split','dev','--model',MODEL,'--output',f'runs/{name}.jsonl']
    if adapter: args += ['--adapter',adapter]
    if n > 1: args += ['--num-candidates',str(n)]
    if filt: args += ['--filter']
    run(*args)


In [ ]:
import json, pandas as pd
names=['template_dev','prompt_dev','sft_dev','prompt_filter_dev','sft_filter_dev']
display(pd.DataFrame([{'method':n, **json.loads(Path(f'runs/{n}.summary.json').read_text())} for n in names])
        [['method','n','exact_span_rate','selected_string_check_rate','string_pass_at_n','mean_candidates_generated']])
print('These automatic scores require human semantic validation.')


## Gate 3 — optional human-audited preferences and DPO

Generate train candidates, download the CSV, fill **every** `human_valid` with true/false after checking facts, scope, preservation, and fluency, then upload the completed file. Only run DPO if enough reliable same-prompt positive/negative pairs exist.


In [ ]:
run('evaluate_generator.py','--data','data/real.jsonl','--split','train','--model',MODEL,
    '--adapter','runs/sft/adapter','--num-candidates','4','--output','runs/sft_train_candidates.jsonl')
run('export_audit.py','--data','data/real.jsonl','--predictions','runs/sft_train_candidates.jsonl',
    '--output','data/sft_train_audit.csv')
files.download('data/sft_train_audit.csv')


**Manual pause:** edit the downloaded CSV yourself. The next cell waits for your completed audit. Do not fill labels from the string-check column.


In [ ]:
uploaded = files.upload()  # completed sft_train_audit.csv
assert 'sft_train_audit.csv' in uploaded
Path('data/sft_train_audit.csv').write_bytes(uploaded['sft_train_audit.csv'])
run('prepare_preferences.py','--data','data/real.jsonl','--audit','data/sft_train_audit.csv',
    '--output','data/train_preferences.jsonl')
run('train_dpo.py','--pairs','data/train_preferences.jsonl','--sft-adapter','runs/sft/adapter',
    '--model',MODEL,'--output','runs/dpo')
run('evaluate_generator.py','--data','data/real.jsonl','--split','dev','--model',MODEL,
    '--adapter','runs/dpo/adapter','--num-candidates','4','--filter',
    '--output','runs/dpo_filter_dev.jsonl')


## Gate 4 — retrieval sanity check and fixed-pool evaluation

The Flask pair only checks code. For a research result, prepare a fixed `retrieval_test.jsonl` with old, current, and same-version distractor documents, then compare all methods on it.


In [ ]:
run('make_retrieval_benchmark.py','--data','data/flask_pilot_audited.jsonl',
    '--split','dev','--output','runs/flask_pair.jsonl')
for method in ('bm25','version_bm25'):
    run('retrieval_eval.py','--benchmark','runs/flask_pair.jsonl','--method',method,
        '--output',f'runs/flask_{method}.json')


For reranker training and a real candidate pool, follow `COLAB_RUNBOOK.md`. Keep final test data untouched until all model and threshold choices are fixed.
